In [ ]:


# Install dependencies
!pip install -q transformers accelerate bitsandbytes qwen-vl-utils
# Optional: flash_attention_2 for better acceleration (uncomment if needed)
# !pip install flash-attn --no-build-isolation

from __future__ import annotations

from pathlib import Path
from typing import Any, List
import json
import re
import time
import gc
import warnings
from collections import Counter
from PIL import Image
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from tqdm.auto import tqdm

# Suppress harmless generation config warnings
warnings.filterwarnings("ignore", message=".*generation flags are not valid.*")

# ============================================================================
# CONFIGURATION - Optimized for A100 40GB GPU
# ============================================================================
MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"
IMAGES_DIR = Path("images")  # Directory with images
OUTPUT_DIR = IMAGES_DIR  # Save JSON files here (same as images or change)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}
MAX_NEW_TOKENS = 256

# GPU Optimization Settings - Optimized for A100 40GB
USE_BF16 = True  # Use bfloat16 for better performance on A100
USE_COMPILE = True  # Use torch.compile for faster inference (PyTorch 2.0+)
ENABLE_FLASH_ATTN = False  # Set to True if flash-attn is installed
# Note: Processing images individually with optimizations for maximum GPU utilization

# ============================================================================
# GPU OPTIMIZATION SETUP
# ============================================================================
print("🚀 Setting up GPU optimizations...")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    # Enable performance optimizations
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
        if hasattr(torch, "set_float32_matmul_precision"):
            torch.set_float32_matmul_precision("high")
        print("✅ TF32 and cuDNN optimizations enabled")
    except Exception as e:
        print(f"⚠️  Could not enable all optimizations: {e}")
    
    # Determine dtype
    if USE_BF16 and torch.cuda.is_bf16_supported():
        torch_dtype = torch.bfloat16
        print("✅ Using bfloat16 precision")
    else:
        torch_dtype = torch.float16
        print("✅ Using float16 precision")
else:
    device = torch.device("cpu")
    torch_dtype = torch.float32
    print("⚠️  No GPU detected, using CPU")

# ============================================================================
# LOAD MODEL WITH OPTIMIZATIONS
# ============================================================================
print(f"\n📦 Loading {MODEL_NAME}...")
try:
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch_dtype,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    
    # Move to device explicitly if needed
    if hasattr(model, "device") and str(model.device) != str(device):
        model = model.to(device)
    
    model.eval()
    
    # Optional: Compile model for faster inference (PyTorch 2.0+)
    if USE_COMPILE and hasattr(torch, "compile"):
        try:
            print("⚡ Compiling model with torch.compile...")
            model = torch.compile(model, mode="reduce-overhead")
            print("✅ Model compiled successfully")
        except Exception as e:
            print(f"⚠️  torch.compile failed (continuing without): {e}")
    
    # Clear cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    print("✅ Model loaded and ready")
    print(f"✅ Model device: {next(model.parameters()).device}")
    print(f"✅ Model dtype: {next(model.parameters()).dtype}")
except Exception as e:
    print(f"❌ Failed to load model: {e}")
    raise


# ============================================================================
# PROMPT BUILDING
# ============================================================================
def build_object_extraction_prompt() -> str:
    """Build prompt for Qwen to extract simple object names only."""
    return """List ALL visible objects in this image as JSON.

Format:
{
    "objects": ["object1", "object2", ...],
    "categories": {
        "signs": [...],
        "accessibility": [...],
        "people": [...],
        "furniture": [...],
        "technology": [...],
        "other": [...]
    },
    "scene_description": "brief description",
    "primary_focus": "main subject"
}

CRITICAL RULES:
- Extract ONLY object names - NO adjectives, colors, shapes, sizes, materials, or descriptions
- Examples: "car" (NOT "red car", "small car", "blue vehicle")
- Examples: "chair" (NOT "wooden chair", "black chair", "office chair")
- Examples: "person" (NOT "tall person", "woman", "man", "people")
- Simple nouns (singular, shortest form)
- NO SYNONYMS: Use "person" (NOT "people", "men", "woman", "male", "female", "human")
- NO GENDER-SPECIFIC: Use "person" for all people
- Keep names SHORT: prefer "car" over "vehicle", "bike" over "bicycle"
- SIGNS: "exit sign", "restroom sign", "elevator sign", "emergency sign"
- ACCESSIBILITY: "wheelchair ramp", "braille sign", "accessible entrance"
- No duplicates or variations
- JSON only, no markdown"""


# ============================================================================
# RESPONSE PARSING
# ============================================================================
def parse_qwen_response(response: str) -> dict[str, Any]:
    """Parse Qwen's JSON response into structured data."""
    def _strip_code_fences(text: str) -> str:
        text = text.strip()
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)
        return text.strip()

    def _extract_balanced_json(text: str) -> str | None:
        """Extract first balanced JSON object."""
        start = text.find("{")
        if start == -1:
            return None
        depth = 0
        in_str = False
        esc = False
        for i in range(start, len(text)):
            ch = text[i]
            if in_str:
                if esc:
                    esc = False
                elif ch == "\\":
                    esc = True
                elif ch == '"':
                    in_str = False
            else:
                if ch == '"':
                    in_str = True
                elif ch == "{":
                    depth += 1
                elif ch == "}":
                    depth -= 1
                    if depth == 0:
                        return text[start:i + 1]
        return None

    try:
        cleaned = _strip_code_fences(response)
        json_str = _extract_balanced_json(cleaned)
        if json_str:
            for attempt in range(2):
                try:
                    data = json.loads(json_str)
                    break
                except json.JSONDecodeError:
                    json_str = json_str.replace("'", '"')
                    json_str = re.sub(r",\s*}", "}", json_str)
                    json_str = re.sub(r",\s*]", "]", json_str)
            else:
                data = None

            if isinstance(data, dict):
                objects = data.get("objects", []) or []
                categories = data.get("categories", {}) or {}
                return {
                    "objects": objects,
                    "categories": categories,
                    "scene_description": str(data.get("scene_description", ""))[:500],
                    "primary_focus": str(data.get("primary_focus", ""))[:200],
                    "num_objects": len(objects),
                    "error": False,
                }
    except Exception as e:
        pass  # Silent fallback

    # Fallback: extract objects from plain text
    objects_list = re.findall(r'["\']([^"\']+)["\']', response)
    return {
        "objects": objects_list[:200],
        "categories": {},
        "scene_description": "",
        "primary_focus": "",
        "num_objects": len(objects_list),
        "error": False,
    }


# ============================================================================
# OPTIMIZED SINGLE IMAGE PROCESSING
# ============================================================================
def process_single_image(image_path: Path, output_dir: Path) -> dict[str, Any]:
    """Process one image with GPU optimizations."""
    try:
        image = Image.open(image_path).convert("RGB")
        prompt = build_object_extraction_prompt()
        
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt}
                ]
            }
        ]
        
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = processor(
            text=[text],
            images=image_inputs if image_inputs else None,
            videos=video_inputs if video_inputs else None,
            padding=True,
            return_tensors="pt"
        )
        
        model_device = next(model.parameters()).device
        inputs = {k: v.to(model_device) if hasattr(v, "to") else v for k, v in inputs.items()}
        
        start_time = time.time()
        with torch.inference_mode():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
            )
        
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs['input_ids'], generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]
        
        elapsed = time.time() - start_time
        
        result = parse_qwen_response(output_text)
        result["image_path"] = str(image_path)
        result["processing_time_seconds"] = round(elapsed, 2)
        
        output_path = output_dir / f"{image_path.stem}.json"
        output_path.write_text(json.dumps(result, indent=2))
        
        # Periodic cache clearing (every 20 images to maximize GPU utilization)
        if torch.cuda.is_available():
            if not hasattr(process_single_image, '_call_count'):
                process_single_image._call_count = 0
            process_single_image._call_count += 1
            if process_single_image._call_count % 20 == 0:
                torch.cuda.empty_cache()
        
        return result
    except Exception as e:
        return {
            "image_path": str(image_path),
            "error": str(e),
            "processing_time_seconds": 0,
        }


# ============================================================================
# MAIN PROCESSING LOOP - BATCH OPTIMIZED
# ============================================================================
if not IMAGES_DIR.exists():
    IMAGES_DIR.mkdir(parents=True, exist_ok=True)
    print(f"⚠️ Created directory: {IMAGES_DIR}")
    print("   Please add images to this directory and re-run this cell")

image_files = sorted([f for f in IMAGES_DIR.iterdir() 
                      if f.suffix.lower() in IMAGE_EXTENSIONS])

if not image_files:
    print(f"⚠️ No images found in {IMAGES_DIR}")
    print(f"   Supported formats: {', '.join(IMAGE_EXTENSIONS)}")
else:
    print(f"✅ Found {len(image_files)} image(s) to process")
    print(f"✅ GPU optimizations: bf16={USE_BF16}, compile={USE_COMPILE}\n")
    
    # Process all images with optimized single-image processing
    print(f"🔄 Processing {len(image_files)} image(s) with GPU optimizations...\n")
    total_start = time.time()
    
    results = []
    
    # Process with progress tracking
    for i, image_path in enumerate(tqdm(image_files, desc="Processing", unit="img"), 1):
        try:
            result = process_single_image(image_path, OUTPUT_DIR)
            
            # FIX: Only add 'json' key on success, never on error
            if "error" in result:
                results.append({
                    "image": image_path.name,
                    "error": result["error"]
                })
            else:
                json_path = OUTPUT_DIR / f"{image_path.stem}.json"
                results.append({
                    "image": image_path.name,
                    "objects": result.get("num_objects", 0),
                    "time": result.get("processing_time_seconds", 0),
                    "json": str(json_path)  # Only add 'json' key on success
                })
        except Exception as e:
            # Handle any unexpected errors - no 'json' key
            results.append({
                "image": image_path.name,
                "error": str(e)
            })
        
        # Progress update every 50 images
        if i % 50 == 0 or i == len(image_files):
            elapsed = time.time() - total_start
            rate = i / elapsed if elapsed > 0 else 0
            remaining = len(image_files) - i
            eta = remaining / rate if rate > 0 else 0
            successful = len([r for r in results if "error" not in r])
            print(f"\n   Progress: {i}/{len(image_files)} ({rate:.2f} img/s, {successful} successful, ETA: {eta:.0f}s)")
            if torch.cuda.is_available():
                memory_used = torch.cuda.memory_allocated(0) / 1e9
                memory_total = torch.cuda.get_device_properties(0).total_memory / 1e9
                print(f"   GPU Memory: {memory_used:.1f} GB / {memory_total:.1f} GB ({memory_used/memory_total*100:.1f}%)")
    
    total_elapsed = time.time() - total_start
    
    # Summary
    print(f"\n{'='*60}")
    print(f"✅ COMPLETED in {total_elapsed:.1f}s ({total_elapsed/len(image_files):.2f}s per image)")
    print(f"   JSON files saved in: {OUTPUT_DIR}")
    print(f"{'='*60}\n")
    
    successful = [r for r in results if "error" not in r]
    failed = [r for r in results if "error" in r]
    
    if successful:
        total_objects = sum(r.get("objects", 0) for r in successful)
        avg_time = sum(r.get("time", 0) for r in successful) / len(successful) if successful else 0
        print(f"✅ Successfully processed: {len(successful)} image(s)")
        print(f"   Total objects detected: {total_objects}")
        print(f"   Average processing time: {avg_time:.2f}s per image")
        print(f"   Throughput: {len(successful)/total_elapsed:.2f} images/second")
    
    if failed:
        print(f"\n❌ Failed: {len(failed)} image(s)")
        for r in failed[:10]:  # Show first 10 failures
            print(f"   • {r['image']}: {r.get('error', 'Unknown error')}")
        if len(failed) > 10:
            print(f"   ... and {len(failed) - 10} more failures")
    
    # GPU memory usage
    if torch.cuda.is_available():
        memory_allocated = torch.cuda.memory_allocated(0) / 1e9
        memory_reserved = torch.cuda.memory_reserved(0) / 1e9
        memory_total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"\n💾 GPU Memory: {memory_allocated:.1f} GB allocated / {memory_reserved:.1f} GB reserved / {memory_total:.1f} GB total")
        print(f"   Utilization: {memory_allocated/memory_total*100:.1f}%")
    
    # List generated JSON files
    json_files = sorted(OUTPUT_DIR.glob("*.json"))
    if json_files:
        print(f"\n📁 Generated JSON files ({len(json_files)}):")
        for json_file in json_files[:5]:
            size_kb = json_file.stat().st_size / 1024
            print(f"   • {json_file.name} ({size_kb:.1f} KB)")
        if len(json_files) > 5:
            print(f"   ... and {len(json_files) - 5} more")



# ============================================================================
# VIEW DETAILED RESULTS (Optional)
# ============================================================================
if 'results' in locals() and results:
    # Find first successful result
    first_success = None
    for r in results:
        if "error" not in r and "json" in r:
            first_success = r
            break
    
    if first_success:
        first_json = first_success.get("json")
        if first_json and Path(first_json).exists():
            try:
                result_data = json.loads(Path(first_json).read_text())
                
                print("\n" + "=" * 60)
                print(f"DETAILED RESULTS: {first_success['image']}")
                print("=" * 60)
                
                objects = result_data.get("objects", [])
                if objects:
                    counts = Counter(objects)
                    
                    print(f"\n📊 Total objects: {len(objects)}")
                    print(f"\n📊 Object counts:")
                    for obj, count in counts.most_common(20):  # Top 20
                        print(f"  {obj}: {count}")
                    
                    if len(objects) <= 50:
                        print(f"\n📋 All objects ({len(objects)}):")
                        for i, obj in enumerate(objects, 1):
                            print(f"  {i:3d}. {obj}")
                    
                    categories = result_data.get("categories", {})
                    if categories:
                        print(f"\n📂 Categories:")
                        for cat, items in categories.items():
                            if items:
                                print(f"  • {cat}: {', '.join(items)}")
                    
                    print(f"\n🎯 Primary focus: {result_data.get('primary_focus', 'N/A')}")
                    print(f"📝 Scene: {result_data.get('scene_description', 'N/A')}")
                print("=" * 60)
            except Exception as e:
                print(f"⚠️  Could not display detailed results: {e}")
